# Grid Search dos modelos supervisionados

Este notebook percorre combinações discretas de hiperparâmetros para Regressão Logística, SVM linear e Random Forest, além dos preprocessadores disponíveis.

A seleção usa F1 macro em validação cruzada estratificada. O conjunto de teste permanece isolado durante a busca e só será avaliado uma vez para os melhores candidatos. Para a amostragem aleatória, consulte `05_Hyperparameter_Search.ipynb`.

## 1. Importar bibliotecas e carregar os modelos existentes

Os estimadores serão montados em `Pipeline`, mantendo cada `ColumnTransformer` dentro da validação cruzada. Assim, vocabulário TF-IDF e demais transformações são ajustados somente nos folds de treino.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory

import joblib
import numpy as np
import pandas as pd
from joblib import Memory
from scipy.stats import loguniform, randint
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, roc_auc_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

In [ ]:
current_directories = (Path.cwd(), *Path.cwd().parents)
data_directory = next(
    (
        candidate
        for current_directory in current_directories
        for candidate in (
            current_directory,
            current_directory / "machine-learning" / "supervised-learning",
            current_directory / "supervised-learning",
        )
        if (candidate / "dados_preparados.pkl").is_file()
        and (candidate / "transformers.pkl").is_file()
    ),
    None,
)

if data_directory is None:
    raise FileNotFoundError(
        "Artefatos não encontrados. Execute primeiro o notebook 01_Data_Prep.ipynb."
    )

X_tr, X_te, y_tr, y_te = joblib.load(data_directory / "dados_preparados.pkl")
configs = joblib.load(data_directory / "transformers.pkl")

print(f"Diretório dos artefatos: {data_directory}")
print(f"Treino: {X_tr.shape}; teste: {X_te.shape}")
print(f"Distribuição das classes no treino: {y_tr.value_counts().to_dict()}")
print(f"Combinações de atributos: {list(configs)}")

Diretório dos artefatos: c:\Users\BLU8CA\OneDrive - Bosch Group\Desktop\PUC\olimpo-fake-news-ai\machine-learning\supervised-learning
Treino: (5760, 15); teste: (1440, 15)
Distribuição das classes no treino: {0: 2880, 1: 2880}
Combinações de atributos: ['word', 'word+char', 'word+meta', 'word+char+meta']


## 2. Configurar a validação e medir a linha de base

A busca compara os mesmos quatro pré-processadores usados nos treinamentos. A validação é estratificada em três folds; os parâmetros-base reproduzem as configurações atuais de Regressão Logística, SVM linear e Random Forest. O teste não participa da seleção.

In [ ]:
PREPROCESSORS = {
    config_name: ColumnTransformer(transformers)
    for config_name, transformers in configs.items()
}
CONFIG_BY_SIGNATURE = {
    tuple(name for name, _, _ in transformers): config_name
    for config_name, transformers in configs.items()
}
CV = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
SCORING = {
    "f1_macro": "f1_macro",
    "accuracy": "accuracy",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "roc_auc": "roc_auc",
}
METRIC_COLUMNS = {
    "f1_macro": "f1_macro_cv",
    "accuracy": "accuracy_cv",
    "precision_macro": "precision_macro_cv",
    "recall_macro": "recall_macro_cv",
    "roc_auc": "roc_auc_cv",
}

MODEL_SPECS = {
    "Regressão Logística": {
        "estimator": LogisticRegression(max_iter=1000, class_weight="balanced"),
        "baseline_params": {"C": 1.0, "class_weight": "balanced"},
        "grid": {
            "clf__C": [0.1, 1.0, 10.0],
            "clf__class_weight": [None, "balanced"],
        },
        "random": {
            "clf__C": loguniform(1e-3, 1e2),
            "clf__class_weight": [None, "balanced"],
        },
    },
    "SVM Linear": {
        "estimator": LinearSVC(class_weight="balanced", random_state=42, max_iter=2000),
        "baseline_params": {"C": 1.0, "class_weight": "balanced"},
        "grid": {
            "clf__C": [0.1, 1.0, 10.0],
            "clf__class_weight": [None, "balanced"],
        },
        "random": {
            "clf__C": loguniform(1e-3, 1e2),
            "clf__class_weight": [None, "balanced"],
        },
    },
    "Random Forest": {
        "estimator": RandomForestClassifier(n_estimators=100, random_state=42),
        "baseline_params": {
            "n_estimators": 100,
            "max_depth": None,
            "min_samples_leaf": 1,
            "max_features": "sqrt",
            "class_weight": None,
        },
        "grid": {
            "clf__n_estimators": [100, 250],
            "clf__max_depth": [None, 30],
            "clf__min_samples_leaf": [1, 3],
            "clf__class_weight": [None, "balanced"],
        },
        "random": {
            "clf__n_estimators": randint(100, 501),
            "clf__max_depth": [None, 10, 20, 30, 50],
            "clf__min_samples_split": randint(2, 21),
            "clf__min_samples_leaf": randint(1, 9),
            "clf__max_features": ["sqrt", 0.5, None],
            "clf__class_weight": [None, "balanced"],
        },
    },
}

RANDOM_SEARCH_ITERATIONS = 16
N_JOBS = 1
CACHE_DIRECTORY = TemporaryDirectory(prefix="fakebr-hyperparameter-search-")
PIPELINE_MEMORY = Memory(location=CACHE_DIRECTORY.name, verbose=0)


def create_pipeline(preprocessor, estimator):
    return Pipeline(
        [("prep", preprocessor), ("clf", clone(estimator))],
        memory=PIPELINE_MEMORY,
    )


def resolve_config_name(preprocessor):
    signature = tuple(name for name, _, _ in preprocessor.transformers)
    return CONFIG_BY_SIGNATURE[signature]


def format_parameter_value(parameter_value):
    if parameter_value is None:
        return "None"
    if isinstance(parameter_value, (float, np.floating)):
        return f"{float(parameter_value):.4g}"
    if isinstance(parameter_value, (int, np.integer)):
        return str(int(parameter_value))
    return str(parameter_value)


def collect_search_rows(search, model_name, search_name):
    cv_results = search.cv_results_
    result_rows = []
    for index, candidate_params in enumerate(cv_results["params"]):
        row = {
            "modelo": model_name,
            "busca": search_name,
            "config": resolve_config_name(candidate_params["prep"]),
            "f1_macro_cv": float(cv_results["mean_test_f1_macro"][index]),
            "f1_macro_cv_std": float(cv_results["std_test_f1_macro"][index]),
            "f1_macro_treino": float(cv_results["mean_train_f1_macro"][index]),
            "overfit_gap": float(
                cv_results["mean_train_f1_macro"][index]
                - cv_results["mean_test_f1_macro"][index]
            ),
            "tempo_medio_fit_s": float(cv_results["mean_fit_time"][index]),
        }
        for metric_name, column_name in METRIC_COLUMNS.items():
            row[column_name] = float(cv_results[f"mean_test_{metric_name}"][index])
            row[f"{column_name}_std"] = float(
                cv_results[f"std_test_{metric_name}"][index]
            )
        row.update(
            {
                f"param_{name.removeprefix('clf__')}": format_parameter_value(value)
                for name, value in candidate_params.items()
                if name.startswith("clf__")
            }
        )
        result_rows.append(row)
    return result_rows

print(f"Validação: {CV.get_n_splits()} folds estratificados")
print(f"Combinações de atributos por modelo: {len(PREPROCESSORS)}")
print(f"Iterações do Random Search por modelo: {RANDOM_SEARCH_ITERATIONS}")

Validação: 3 folds estratificados
Combinações de atributos por modelo: 4
Iterações do Random Search por modelo: 16


In [ ]:
BASELINE_ROWS = []
for model_name, model_spec in MODEL_SPECS.items():
    for config_name, preprocessor in PREPROCESSORS.items():
        fold_scores = cross_validate(
            create_pipeline(preprocessor, model_spec["estimator"]),
            X_tr,
            y_tr,
            cv=CV,
            scoring=SCORING,
            return_train_score=True,
            n_jobs=N_JOBS,
            error_score="raise",
        )
        result_row = {
            "modelo": model_name,
            "busca": "Baseline",
            "config": config_name,
            "f1_macro_treino": float(fold_scores["train_f1_macro"].mean()),
            "tempo_medio_fit_s": float(fold_scores["fit_time"].mean()),
        }
        for metric_name, column_name in METRIC_COLUMNS.items():
            test_scores = fold_scores[f"test_{metric_name}"]
            result_row[column_name] = float(test_scores.mean())
            result_row[f"{column_name}_std"] = float(test_scores.std())
        result_row["overfit_gap"] = (
            result_row["f1_macro_treino"] - result_row["f1_macro_cv"]
        )
        result_row.update(
            {
                f"param_{name}": format_parameter_value(value)
                for name, value in model_spec["baseline_params"].items()
            }
        )
        BASELINE_ROWS.append(result_row)

BASELINE_DF = pd.DataFrame(BASELINE_ROWS)
BASELINE_LEADERBOARD = (
    BASELINE_DF.sort_values("f1_macro_cv", ascending=False)
    .groupby("modelo", sort=False)
    .head(1)
    .sort_values("f1_macro_cv", ascending=False)
)
display(
    BASELINE_LEADERBOARD[
        [
            "modelo",
            "config",
            "f1_macro_cv",
            "f1_macro_cv_std",
            "accuracy_cv",
            "roc_auc_cv",
            "overfit_gap",
        ]
    ].reset_index(drop=True)
)

c:\Users\BLU8CA\OneDrive - Bosch Group\Desktop\PUC\olimpo-fake-news-ai\machine-learning\.venv\Lib\site-packages\sklearn\svm\_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\BLU8CA\OneDrive - Bosch Group\Desktop\PUC\olimpo-fake-news-ai\machine-learning\.venv\Lib\site-packages\sklearn\svm\_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\BLU8CA\OneDrive - Bosch Group\Desktop\PUC\olimpo-fake-news-ai\machine-learning\.venv\Lib\site-packages\sklearn\svm\_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\BLU8CA\OneDrive - Bosch Group\Desktop\PUC\olimpo-fake-news-ai\machine-learning\.venv\Lib\site-packages\sklearn\svm\_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\BLU8CA\OneDrive - Bosch Gro

,modelo,config,f1_macro_cv,f1_macro_cv_std,accuracy_cv,roc_auc_cv,overfit_gap
0,SVM Linear,word+char+meta,0.939741,0.003202,0.939757,0.984193,0.060085
1,Random Forest,word+meta,0.924994,0.005622,0.925000,0.979103,0.075006
2,Regressão Logística,word+char+meta,0.916279,0.004441,0.916319,0.968045,0.052901


## 3. Grid Search

Para cada modelo, o Grid Search percorre todas as combinações discretas de hiperparâmetros e preprocessadores disponíveis. Cada candidato é avaliado nos três folds estratificados; TF-IDF e classificador são ajustados em cada fold. Para a amostragem aleatória, consulte `05_Hyperparameter_Search.ipynb`.

In [ ]:
from sklearn.model_selection import ParameterGrid

GRID_CANDIDATES = {
    model_name: len(
        list(
            ParameterGrid(
                {"prep": list(PREPROCESSORS.values()), **model_spec["grid"]}
            )
        )
    )
    for model_name, model_spec in MODEL_SPECS.items()
}
GRID_FITS = sum(GRID_CANDIDATES.values()) * CV.get_n_splits()

print(f"Grid Search: {GRID_CANDIDATES} candidatos por modelo")
print(f"Ajustes estimados nos folds: {GRID_FITS}")

In [ ]:
if "taxa_stop_words" not in X_tr.columns:
    raise ValueError(
        "Os artefatos não incluem taxa_stop_words. Execute novamente o notebook 01_Data_Prep.ipynb."
    )

has_taxa_in_metadata = any(
    "taxa_stop_words" in columns
    for transformers in configs.values()
    for name, _, columns in transformers
    if name == "meta"
)
if not has_taxa_in_metadata:
    raise ValueError("Os preprocessadores carregados não incluem taxa_stop_words.")

SEARCH_ROWS = []
SEARCH_OBJECTS = {}

for model_name, model_spec in MODEL_SPECS.items():
    estimator = create_pipeline(PREPROCESSORS["word"], model_spec["estimator"])
    search = GridSearchCV(
        estimator=estimator,
        param_grid={
            "prep": list(PREPROCESSORS.values()),
            **model_spec["grid"],
        },
        scoring=SCORING,
        refit="f1_macro",
        cv=CV,
        return_train_score=True,
        n_jobs=N_JOBS,
        error_score="raise",
    )
    search.fit(X_tr, y_tr)
    SEARCH_OBJECTS[(model_name, "Grid Search")] = search
    SEARCH_ROWS.extend(collect_search_rows(search, model_name, "Grid Search"))
    print(
        f"{model_name} | Grid Search: F1 macro CV={search.best_score_:.4f}; "
        f"config={resolve_config_name(search.best_params_['prep'])}"
    )

SEARCH_RESULTS_DF = pd.DataFrame(SEARCH_ROWS)
BEST_SEARCH_DF = (
    SEARCH_RESULTS_DF.sort_values("f1_macro_cv", ascending=False)
    .groupby("modelo", sort=False)
    .head(1)
)
COMPARISON_DF = BEST_SEARCH_DF.copy()
if "BASELINE_LEADERBOARD" in globals():
    COMPARISON_DF = pd.concat([BASELINE_LEADERBOARD, BEST_SEARCH_DF], ignore_index=True)

comparison_columns = [
    "modelo",
    "busca",
    "config",
    "f1_macro_cv",
    "f1_macro_cv_std",
    "accuracy_cv",
    "precision_macro_cv",
    "recall_macro_cv",
    "roc_auc_cv",
    "overfit_gap",
]
display(
    COMPARISON_DF[comparison_columns]
    .sort_values(["modelo", "f1_macro_cv"], ascending=[True, False])
    .reset_index(drop=True)
)

Regressão Logística | Grid Search: F1 macro CV=0.9366; config=word+char+meta
Regressão Logística | Random Search: F1 macro CV=0.9245; config=word+char


c:\Users\BLU8CA\OneDrive - Bosch Group\Desktop\PUC\olimpo-fake-news-ai\machine-learning\.venv\Lib\site-packages\sklearn\svm\_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\BLU8CA\OneDrive - Bosch Group\Desktop\PUC\olimpo-fake-news-ai\machine-learning\.venv\Lib\site-packages\sklearn\svm\_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\BLU8CA\OneDrive - Bosch Group\Desktop\PUC\olimpo-fake-news-ai\machine-learning\.venv\Lib\site-packages\sklearn\svm\_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\BLU8CA\OneDrive - Bosch Group\Desktop\PUC\olimpo-fake-news-ai\machine-learning\.venv\Lib\site-packages\sklearn\svm\_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\BLU8CA\OneDrive - Bosch Gro

SVM Linear | Grid Search: F1 macro CV=0.9413; config=word+char+meta


c:\Users\BLU8CA\OneDrive - Bosch Group\Desktop\PUC\olimpo-fake-news-ai\machine-learning\.venv\Lib\site-packages\sklearn\svm\_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\BLU8CA\OneDrive - Bosch Group\Desktop\PUC\olimpo-fake-news-ai\machine-learning\.venv\Lib\site-packages\sklearn\svm\_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\BLU8CA\OneDrive - Bosch Group\Desktop\PUC\olimpo-fake-news-ai\machine-learning\.venv\Lib\site-packages\sklearn\svm\_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\BLU8CA\OneDrive - Bosch Group\Desktop\PUC\olimpo-fake-news-ai\machine-learning\.venv\Lib\site-packages\sklearn\svm\_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
c:\Users\BLU8CA\OneDrive - Bosch Gro

SVM Linear | Random Search: F1 macro CV=0.9399; config=word+char+meta
Random Forest | Grid Search: F1 macro CV=0.9333; config=word+meta


In [2]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(11, 6))
sns.barplot(
    data=COMPARISON_DF,
    x="f1_macro_cv",
    y="modelo",
    hue="busca",
)
plt.title("F1 macro por modelo e estratégia de busca")
plt.xlabel("F1 macro médio na validação cruzada")
plt.ylabel("Modelo")
plt.xlim(0, 1)
plt.grid(axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=SEARCH_RESULTS_DF,
    x="tempo_medio_fit_s",
    y="f1_macro_cv",
    hue="busca",
    style="modelo",
    size="f1_macro_cv_std",
    sizes=(30, 180),
    alpha=0.8,
)
plt.title("Desempenho versus custo por candidato")
plt.xlabel("Tempo médio de ajuste por fold (s)")
plt.ylabel("F1 macro médio na validação cruzada")
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

# Nos modelos lineares, C é numérico e foi amostrado em escala logarítmica.
linear_results = SEARCH_RESULTS_DF.loc[
    SEARCH_RESULTS_DF["modelo"].isin(["Regressão Logística", "SVM Linear"])
].copy()
linear_results["param_C_num"] = pd.to_numeric(linear_results["param_C"], errors="coerce")

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for axis, model_name in zip(axes, ["Regressão Logística", "SVM Linear"]):
    model_results = linear_results.loc[linear_results["modelo"] == model_name]
    sns.scatterplot(
        data=model_results,
        x="param_C_num",
        y="f1_macro_cv",
        hue="config",
        style="busca",
        alpha=0.8,
        ax=axis,
    )
    axis.set_xscale("log")
    axis.set_title(model_name)
    axis.set_xlabel("C (escala logarítmica)")
    axis.grid(True, linestyle="--", alpha=0.4)
axes[0].set_ylabel("F1 macro médio na validação cruzada")
fig.tight_layout()
plt.show()

NameError: name 'COMPARISON_DF' is not defined

<Figure size 1100x600 with 0 Axes>

In [ ]:
PARAMETER_EFFECT_ROWS = []
parameter_columns = [
    column for column in SEARCH_RESULTS_DF.columns
    if column.startswith("param_")
]

for model_name, model_results in SEARCH_RESULTS_DF.groupby("modelo", sort=False):
    for parameter_column in parameter_columns:
        observed = model_results[parameter_column].dropna()
        if observed.empty:
            continue

        numeric_values = pd.to_numeric(observed, errors="coerce")
        is_numeric = numeric_values.notna().all()
        values = numeric_values if is_numeric else observed.astype(str)
        analysis_data = model_results.loc[observed.index, ["f1_macro_cv"]].copy()

        if is_numeric and values.nunique() > 8:
            analysis_data["faixa"] = pd.qcut(
                values,
                q=min(4, values.nunique()),
                duplicates="drop",
            ).astype(str)
        else:
            analysis_data["faixa"] = values.astype(str).to_numpy()

        summary = (
            analysis_data.groupby("faixa", observed=True)["f1_macro_cv"]
            .agg(media="mean", desvio="std", candidatos="count")
            .reset_index()
        )
        summary.insert(0, "hiperparametro", parameter_column.removeprefix("param_"))
        summary.insert(0, "modelo", model_name)
        PARAMETER_EFFECT_ROWS.extend(summary.to_dict("records"))

PARAMETER_EFFECT_DF = pd.DataFrame(PARAMETER_EFFECT_ROWS)
display(
    PARAMETER_EFFECT_DF.sort_values(
        ["modelo", "hiperparametro", "media"],
        ascending=[True, True, False],
    ).reset_index(drop=True)
)

print(
    "Interpretação: compare as médias dentro do mesmo modelo e hiperparâmetro. "
    "Faixas numéricas resumem valores contínuos; candidatos e espaços diferentes "
    "entre estratégias não representam uma comparação controlada."
)